## marketing pdf and generate json

In [1]:
import os
import fitz  # PyMuPDF for PDF processing
from PIL import Image
import json
import google.generativeai as genai
from dotenv import load_dotenv
import base64
import time
import numpy as np
import cv2

# === Load API Key ===
load_dotenv()
model_name = "gemini-2.5-pro"
genai.configure(api_key=os.getenv("GOOGLE_GEMINI_API"))
model = genai.GenerativeModel(model_name)

# === Prompt for Gemini ===
PROMPT = """
Carefully extract all text content from the PDF (Ignore any template, header or headings of the page, footer, or decorative elements such as 'Date', 'Page', General instructions , Time Allowed ), maintaining the exact order and formatting as it appears.
Preserve all mathematical equations, formulas, and special characters exactly as they appear in the PDF.
Do not add any headers, descriptions, or labels to the output.

Output only the extracted text content in the following format for an example:

[
{
question_number: 1,
Section_number: A,
Question_ocr_text: 'This is the PV curve <diagram_1>',
pages: [2],
Mark:
}
]
Explanation of the JSON fields:
question_number: The identifier for the question (e.g., 1, 2, etc.).
section_number: The section to which the question belongs (e.g., "A", "B").
question_ocr_text: The actual extracted text of the question from the PDF, including any references to diagrams (e.g., <diagram_1>).
pages: A list of page numbers where the question is located.
mark: The mark/score for the question (e.g., 5, 4, etc.).
"""

# === Load PDF, Convert Pages to Images ===
def pdf_to_images(pdf_path, output_folder):
    doc = fitz.open(pdf_path)
    images = []
    num_pages = doc.page_count
    for i in range(num_pages):
        page = doc.load_page(i)
        pix = page.get_pixmap(dpi=300)
        img_path_default = os.path.join(output_folder, f"page_{i + 1}.jpeg")
        pix.save(img_path_default)
        images.append(img_path_default)
    return images, num_pages

# === Load Base64 Images ===
def load_base64_images(folder_path, num_pages):
    b64_list = []
    for i in range(num_pages):
        img_filename_default = f"page_{i + 1}.jpeg"  # Old naming convention, without DIM
        img_path_default = os.path.join(folder_path, img_filename_default)

        if os.path.exists(img_path_default):
            path = img_path_default
        else:
            print(f"❌ Image {img_filename_default} not found in {folder_path}")
            continue

        with open(path, "rb") as f:
            b64_list.append(base64.b64encode(f.read()).decode())
    return b64_list

# === Batch send to Gemini ===
def send_to_gemini(base64_images):
    try:
        response = model.generate_content([PROMPT] + base64_images)  # Batch processing
        raw = response.text.strip()
        cleaned = raw.strip('```json').strip('```').strip()
        parsed = json.loads(cleaned)
        return parsed
    except Exception as e:
        print(f"❌ Failed to process images: {e}")
        return None

# === Main Process ===
def main(pdf_file_path, output_folder):
    # Ensure the output folder exists
    os.makedirs(output_folder, exist_ok=True)

    # Step 1: Convert PDF to images
    images, num_pages = pdf_to_images(pdf_file_path, output_folder)

    # Step 2: Save images without resizing
    for page_num in range(len(images)):
        img_path = images[page_num]
        original_image = Image.open(img_path)  # Open the original image
        
        # Save the original image (No resizing)
        img_filename_default = f"page_{page_num + 1}.jpeg"
        original_image.save(os.path.join(output_folder, img_filename_default))

    # Step 3: Load base64 images
    images_b64 = load_base64_images(output_folder, len(images))

    # Step 4: Send to Gemini for OCR
    results = send_to_gemini(images_b64)

    if results:
        output_json_filename = f"output.json"
        json_path = os.path.join(output_folder, output_json_filename)

        # Save the OCR results
        with open(json_path, "w") as f:
            json.dump(results, f, indent=3)

        print(f"OCR results saved to {json_path}")
    else:
        print("❌ No results from Gemini OCR")

# Running the process with a given PDF file and output folder
pdf_file_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/mark/Science-SQP.pdf"  # Replace with your PDF file path
output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/mark"  # Replace with your output folder path
main(pdf_file_path, output_folder)


❌ Failed to process images: 400 The input token count (10400612) exceeds the maximum number of tokens allowed (1048576).
❌ No results from Gemini OCR


In [2]:
import os
import fitz  # PyMuPDF for PDF processing
from PIL import Image
import json
import google.generativeai as genai
from dotenv import load_dotenv
import base64
import time
import numpy as np
import cv2

# === Load API Key ===
load_dotenv()
model_name = "gemini-2.5-pro"
genai.configure(api_key=os.getenv("GOOGLE_GEMINI_API"))
model = genai.GenerativeModel(model_name)

# === Prompt for Gemini ===

# === Load PDF, Convert Pages to Images ===
def pdf_to_images(pdf_path, output_folder):
    doc = fitz.open(pdf_path)
    images = []
    num_pages = doc.page_count
    for i in range(num_pages):
        page = doc.load_page(i)
        pix = page.get_pixmap(dpi=300)
        img_path_default = os.path.join(output_folder, f"page_{i + 1}.jpeg")
        pix.save(img_path_default)
        images.append(img_path_default)
    return images, num_pages

# === Load Base64 Images ===
def load_base64_images(folder_path, num_pages):
    b64_list = []
    for i in range(num_pages):
        img_filename_default = f"page_{i + 1}.jpeg"  # Old naming convention, without DIM
        img_path_default = os.path.join(folder_path, img_filename_default)

        if os.path.exists(img_path_default):
            path = img_path_default
        else:
            print(f"❌ Image {img_filename_default} not found in {folder_path}")
            continue

        with open(path, "rb") as f:
            b64_list.append(base64.b64encode(f.read()).decode())
    return b64_list

# === Batch send to Gemini (Split in smaller chunks) ===
def send_to_gemini_batch(base64_images, batch_size=5):  # Default batch size of 5
    results = []
    num_batches = len(base64_images) // batch_size + (1 if len(base64_images) % batch_size != 0 else 0)

    for i in range(num_batches):
        batch = base64_images[i * batch_size : (i + 1) * batch_size]
        try:
            response = model.generate_content([PROMPT] + batch)  # Send the batch
            raw = response.text.strip()
            cleaned = raw.strip('```json').strip('```').strip()
            parsed = json.loads(cleaned)
            results.extend(parsed)  # Append results
        except Exception as e:
            print(f"❌ Failed to process batch {i+1}: {e}")
            continue
    
    return results

# === Main Process ===
def main(pdf_file_path, output_folder):
    # Ensure the output folder exists
    os.makedirs(output_folder, exist_ok=True)

    # Step 1: Convert PDF to images
    images, num_pages = pdf_to_images(pdf_file_path, output_folder)

    # Step 2: Save images without resizing
    for page_num in range(len(images)):
        img_path = images[page_num]
        original_image = Image.open(img_path)  # Open the original image
        
        # Save the original image (No resizing)
        img_filename_default = f"page_{page_num + 1}.jpeg"
        original_image.save(os.path.join(output_folder, img_filename_default))

    # Step 3: Load base64 images
    images_b64 = load_base64_images(output_folder, len(images))

    # Step 4: Send to Gemini for OCR (in batches)
    results = send_to_gemini_batch(images_b64, batch_size=5)  # Set batch size as needed

    if results:
        output_json_filename = f"output.json"
        json_path = os.path.join(output_folder, output_json_filename)

        # Save the OCR results
        with open(json_path, "w") as f:
            json.dump(results, f, indent=3)

        print(f"OCR results saved to {json_path}")
    else:
        print("❌ No results from Gemini OCR")

# Running the process with a given PDF file and output folder
pdf_file_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/mark/Science-SQP.pdf"  # Replace with your PDF file path
output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/mark"  # Replace with your output folder path
main(pdf_file_path, output_folder)


❌ Failed to process batch 1: 400 The input token count (3214956) exceeds the maximum number of tokens allowed (1048576).
❌ Failed to process batch 2: 400 The input token count (3310517) exceeds the maximum number of tokens allowed (1048576).
❌ Failed to process batch 3: 400 The input token count (2967022) exceeds the maximum number of tokens allowed (1048576).


KeyboardInterrupt: 